# 課題2：住宅価格の回帰モデルの作成

本課題では課題1と同じデータを用いてデータ分析の流れを確認します。各セルに入っているコメントの下に、実行するコードを記入してください。わからない場合は、ここまでのレッスン内容や各種ライブラリの公式ドキュメントを参照しましょう。

## 1. 必要なライブラリの読み込み

In [5]:
# 必要なライブラリの読み込み（変更しないでください）
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## 2. データの読み込み

CSVファイル *iowa_ames_housing_price.csv* を読み込み、内容を確認します。

In [13]:
# データを変数datasetに読み込む
dataset = pd.read_csv('iowa_ames_housing_price.csv')

In [14]:
# データの最初の5行を表示
dataset.head()

,Order,area,price,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,1710,208500,60,RL,65.0,8450,Pave,NaN,Reg,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,1262,181500,20,RL,80.0,9600,Pave,NaN,Reg,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,1786,223500,60,RL,68.0,11250,Pave,NaN,IR1,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,1717,140000,70,RL,60.0,9550,Pave,NaN,IR1,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,2198,250000,60,RL,84.0,14260,Pave,NaN,IR1,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


## 3. 目的変数と説明変数の選択

ここでは、以下の列を使用します。

- 目的変数： `price`
- 説明変数： `OverallQual`, `area`, `X2ndFlrSF`, `BsmtFinSF1`,  `OverallCond`

In [16]:
# 目的変数の列名（変更しないでください）
target_col = 'price'

# 説明変数の列名（変更しないでください）
feature_cols = ['OverallQual', 'area', 'X2ndFlrSF', 'BsmtFinSF1', 'OverallCond']

In [ ]:
# target_col と feature_cols を使用して dataset より目的変数と説明変数に該当する列を取得し、
# numpy 配列に変換したものを変数 Y と X に格納する

# Y:目的変数に該当する列
Y = dataset[target_col].to_numpy()

# X:説明変数に該当する列
X = dataset[feature_cols].to_numpy()

## 4. データの分割

この課題では、ホールドアウト法でデータを分割します。

In [ ]:
# X と Y を 機械学習用データとテストデータに7:3で分ける(X_train, X_test, Y_train, Y_test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [22]:
# 機械学習用データを、学習データと検証データに7:3で分ける(X_train, X_valid, Y_train, Y_valid)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=42)

NameError: name 'X_train' is not defined

## 5. モデルの作成から評価まで

線形回帰モデルと、回帰木、ランダムフォレストの3つのモデルを作成して比較します。それぞれのモデルで以下を行います。
- モデルの作成
- 学習データによる学習
- 検証データによる予測
- MSEの算出

In [ ]:
# 線形回帰モデルを作成し、学習・予測を実施して、MSEを算出する
linear_reg = LinearRegression()
linear_reg.fit(X_train, Y_train)
linear_pred = linear_reg.predict(X_valid)
linear_mse = mean_squared_error(Y_valid, linear_pred)
linear_mse

In [ ]:
# 回帰木のモデルを作成し、学習・予測を実施して、MSEを算出する
decision_tree = DecisionTreeRegressor(random_state=42)
decision_tree.fit(X_train, Y_train)
decision_tree_pred = decision_tree.predict(X_valid)
decision_tree_mse = mean_squared_error(Y_valid, decision_tree_pred)
decision_tree_mse

In [ ]:
# ランダムフォレストのモデルを作成し、学習・予測を実施して、MSEを算出する
random_forest = RandomForestRegressor(random_state=42)
random_forest.fit(X_train, Y_train)
random_forest_pred = random_forest.predict(X_valid)
random_forest_mse = mean_squared_error(Y_valid, random_forest_pred)
random_forest_mse

## 6. テストデータによる汎化性能の確認

3つの中でもっともMSEの値が良かったモデルについて、テストデータで汎化性能を確認しましょう。

In [ ]:
# テストデータを使って予測を行いMSEを算出
model_mse = {
    'LinearRegression': linear_mse,
    'DecisionTreeRegressor': decision_tree_mse,
    'RandomForestRegressor': random_forest_mse
}

best_model_name = min(model_mse, key=model_mse.get)

if best_model_name == 'LinearRegression':
    best_model = linear_reg
elif best_model_name == 'DecisionTreeRegressor':
    best_model = decision_tree
else:
    best_model = random_forest

test_pred = best_model.predict(X_test)
test_mse = mean_squared_error(Y_test, test_pred)
test_mse

## 7. グラフによる確認

説明変数が多い場合、「説明変数で目的変数を正しく予測できているか」を可視化することは困難です。そこで「正解と予測値」とを比較することで、予測の精度を可視化してみましょう。

### 散布図による可視化

X軸(横方向)に `テストデータの正解の値` 、Y軸(縦方向)に `予測値` を使って散布図を作成してみましょう。全体的に右肩上がりで、直線上にデータが並んでいれば、予測が行なえていると判断できます。

In [ ]:
# X軸:正解の値、Y軸:予測値で散布図を作成
plt.figure(figsize=(6, 6))
plt.scatter(Y_test, test_pred)
plt.xlabel('Actual value')
plt.ylabel('Predicted value')
plt.title('Actual vs Predicted')
plt.show()

### ヒストグラムによる可視化

「誤差率＝正解の値と予測値との差の割合」をヒストグラムで確認してみましょう。誤差率は以下で計算できます。

`（正解の値－予測値）/ 正解の値`

併せて、誤差率の平均値、標準偏差も出力しましょう。平均値は `numpy` の `mean` 関数、標準偏差は `std` 関数で取得できます。

なお、ヒストグラムを描く際は`matplotlib.pyplot`の`hist`関数が使えます。matplotlib.pyplotはpltという名前をつけているはずなので、ヒストグラム化したいデータを`plt.hist()`の引数に渡すことでヒストグラムを描画できます。

In [ ]:
# 実際の成約価格と予測価格の誤差率をヒストグラムで表示
error_rate = (Y_test - test_pred) / Y_test
plt.hist(error_rate, bins=20)
plt.xlabel('Error rate')
plt.ylabel('Frequency')
plt.title('Distribution of Error Rate')
plt.show()

# 平均値
error_rate_mean = np.mean(error_rate)
error_rate_mean

# 標準偏差
error_rate_std = np.std(error_rate)
error_rate_std